## Overview

This notebook is intended as a quick demonstration on how to write data to IDR Snowflake. For the purposes of this guide, we assume:
1. You're planning to execute this notebook locally, on a CMS-issued laptop.
2. You have all necessary job codes to access the IDR and any relevant data assets.
3. You've already installed the [Python Snowflake connector](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-install).
4. You have write access to an IDR VDM.
5. Your client laptop or VM image has a web browser installed.

For the purposes of this notebook, we leave the Snowflake role unspecified, which implicitly uses your default IDR Snowflake role and warehouse. If you need to specify a role and/or warehouse for the queries you wish to run, see the README for details.

## Creating a table by executing an arbitrary SQL query

In [ ]:
import snowflake.connector

# update to include your EUA ID
# optionally add other parameters
ctx = snowflake.connector.connect(
          account="cms-idr.privatelink",
          user="<your_eua>",
          authenticator="externalbrowser"
    )

In [ ]:
# update to include your ADM/VDM and the table name that you'd like to write to
with ctx.cursor() as cur:
    cur.execute(
    """
        CREATE TABLE IDRC_PRD.<your_adm_or_vdm>.<your_table> AS
            SELECT * FROM IDRC_PRD.CMS_VDM_VIEW_MDCR_PRD.V2_MDCR_BENE
            LIMIT 10
    """)

## Creating a table by uploading a Pandas dataframe

In [ ]:
import pandas as pd

# update with a path to a CSV on your local machine, and specify the ADM/VDM you want to use under the schema argument below
df = pd.read_csv("path/to/your/file.csv")
snowflake.write_pandas(
    conn = ctx, 
    df = df, 
    table_name = "<your_table>",
    database = "IDRC_PRD",
    schema = "<your_adm_or_vdm>"
)

## Creating a table by executing a SQL query via Snowpark

For this subsection, we additionally assume that you've already set up the [Python Snowpark API](https://docs.snowflake.com/en/developer-guide/snowpark/python/setup).

In [ ]:
# Create a snowpark session object for submitting commands against.
from snowflake.snowpark.session import Session

def snowpark_session_create():
	connection_params = {
			"account": 'cms-idr.privatelink',
			"user": '<your_eua>',
			"authenticator":'externalbrowser',
			"warehouse": "IDRC_PRD_COMM_WH"
		}
	session = Session.builder.configs(connection_params).create()
	return session

session = snowpark_session_create()

# Define a test query in a Snowflake Dataframe
df = session.sql("""   
	select geo_zip5_cd
	, count(1)
	from idrc_prd.cms_vdm_view_mdcr_prd.v2_mdcr_bene
	where idr_ltst_trans_flg = 'Y'
	and idr_trans_obslt_ts > current_date()
	and (bene_death_dt is null or bene_death_dt > current_date())
	group by geo_zip5_cd
""")

# Execute the query in a Snowflake dataframe (result stays in database memory)
df.collect()

In [ ]:
# Write Snowflake dataframe to database as a table.
table_name = "<your_adm>.<your_table>"
df.write.mode("overwrite").save_as_table(table_name) # Final table 